# Planetary-dominated eddies: background tilt direction

## Hypothesis

Where the planetary PV-gradient contribution dominates, the observed tilt should reveal the direction in which each polarity naturally tends to tilt. The working vector hypothesis is:

1. AEs and CEs share a zonal tilt component with approximately the same sign and magnitude.
2. Their meridional components have comparable magnitudes but opposite signs: AEs equatorward and CEs poleward.
3. The zonal sign is estimated from the data rather than assumed, because preliminary notes disagree on whether it is eastward or westward.

Direction is analysed only when `TiltDis >= 5 km`. Results are summarised by eddy as well as by daily observation to avoid treating every day of a long track as an independent replicate.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (ANALYSIS_ROOT, CASE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from case_study_tools import PVAlignmentConfig, add_pv_alignment_diagnostics

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 60)

## 1. Load data and define the planetary-dominated population

In [ ]:
DOMINANCE_FACTOR = 2.0
MIN_DEPTH_m = 3000.0
MIN_TILT_km = 5.0

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = tilt.add_region_labels(df, grid)
df = tilt.add_pv_gradient_terms(df, grid, core_mean=True)

config = PVAlignmentConfig(
    dominance_factor=DOMINANCE_FACTOR,
    min_open_ocean_depth_m=MIN_DEPTH_m,
    min_tilt_distance_km=MIN_TILT_km,
)
d = add_pv_alignment_diagnostics(df, config)
planetary = d[d.open_ocean_planetary & d.direction_valid].copy()

# Compass bearing: 0° = north and 90° = east.
planetary['tilt_x_km'] = planetary.TiltDis * np.sin(np.deg2rad(planetary.TiltDir))
planetary['tilt_y_km'] = planetary.TiltDis * np.cos(np.deg2rad(planetary.TiltDir))

display(planetary.groupby('Cyc').agg(
    observations=('Eddy', 'size'), eddies=('Eddy', 'nunique'),
    median_depth_m=('h', 'median'), median_tilt_km=('TiltDis', 'median')
).round(2))

## 2. Spatial and directional context

The maps check that the selected population is genuinely offshore. The polar distributions show bearings, while the component plots directly test the vector hypothesis.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
for ax, cyc, cmap in zip(axs, ['AE', 'CE'], ['Reds', 'Blues']):
    part = planetary[planetary.Cyc == cyc]
    bath = ax.contourf(grid.X_grid, grid.Y_grid, np.where(grid.mask_rho, grid.h / 1e3, np.nan), cmap='Greys_r')
    ax.scatter(part.xc, part.yc, s=4, c=part.TiltDis, cmap=cmap, alpha=.55, edgecolors='none')
    ax.contour(grid.X_grid, grid.Y_grid, grid.h, levels=[MIN_DEPTH_m], colors='gold')
    tilt.lat_lon_contours(ax, grid)
    ax.set(title=cyc, xlabel='x (km)', aspect='equal')
axs[0].set_ylabel('y (km)')
fig.colorbar(bath, ax=axs, label='Depth (km)', fraction=.025, pad=.02);

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': 'polar'}, constrained_layout=True)
bins = np.deg2rad(np.arange(0, 361, 15))
for ax, cyc, color in zip(axs, ['AE', 'CE'], ['tab:red', 'tab:blue']):
    theta = np.deg2rad(planetary.loc[planetary.Cyc == cyc, 'TiltDir'])
    ax.hist(theta, bins=bins, weights=np.ones(len(theta)) * 100 / len(theta), color=color, alpha=.75)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title(f'{cyc} tilt bearing')
    ax.set_ylabel('Observations (%)')

## 3. Eddy-level vector components

First average each component within each eddy. These eddy means, rather than individual days, form the principal sampling units. Positive x is eastward and positive y is northward.

In [ ]:
eddy_vectors = (planetary.groupby(['Cyc', 'Eddy'], as_index=False)
                .agg(x_km=('tilt_x_km', 'mean'), y_km=('tilt_y_km', 'mean'),
                     tilt_km=('TiltDis', 'mean'), observations=('Day', 'size')))
display(eddy_vectors.groupby('Cyc').agg(
    eddies=('Eddy', 'size'), mean_x_km=('x_km', 'mean'), median_x_km=('x_km', 'median'),
    mean_y_km=('y_km', 'mean'), median_y_km=('y_km', 'median')
).round(2))

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    part = eddy_vectors[eddy_vectors.Cyc == cyc]
    axs[0].hist(part.x_km, bins='fd', density=True, histtype='step', lw=2, color=color, label=cyc)
    axs[1].hist(part.y_km, bins='fd', density=True, histtype='step', lw=2, color=color, label=cyc)
for ax, label in zip(axs, ['Mean zonal tilt per eddy (km; east +)', 'Mean meridional tilt per eddy (km; north +)']):
    ax.axvline(0, color='0.4', lw=.8)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
axs[0].legend(frameon=False);

## 4. Eddy-bootstrap uncertainty and explicit hypothesis contrasts

The shared-zonal hypothesis predicts `AE_x - CE_x ≈ 0`. The opposite-meridional hypothesis predicts `AE_y + CE_y ≈ 0`, alongside AE and CE meridional means having opposite signs. Bootstrap intervals are resampled over eddies.

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 10_000
ae = eddy_vectors[eddy_vectors.Cyc == 'AE']
ce = eddy_vectors[eddy_vectors.Cyc == 'CE']
boot = np.empty((N_BOOT, 4))
for i in range(N_BOOT):
    a = ae.sample(len(ae), replace=True, random_state=rng.integers(2**32 - 1))
    c = ce.sample(len(ce), replace=True, random_state=rng.integers(2**32 - 1))
    boot[i] = [a.x_km.mean(), c.x_km.mean(), a.y_km.mean(), c.y_km.mean()]

contrasts = pd.DataFrame({
    'quantity': ['AE mean x', 'CE mean x', 'AE x − CE x', 'AE mean y', 'CE mean y', 'AE y + CE y'],
    'estimate_km': [ae.x_km.mean(), ce.x_km.mean(), ae.x_km.mean()-ce.x_km.mean(),
                    ae.y_km.mean(), ce.y_km.mean(), ae.y_km.mean()+ce.y_km.mean()],
    'low_95_km': np.percentile(np.c_[boot[:,0], boot[:,1], boot[:,0]-boot[:,1],
                                      boot[:,2], boot[:,3], boot[:,2]+boot[:,3]], 2.5, axis=0),
    'high_95_km': np.percentile(np.c_[boot[:,0], boot[:,1], boot[:,0]-boot[:,1],
                                       boot[:,2], boot[:,3], boot[:,2]+boot[:,3]], 97.5, axis=0),
})
display(contrasts.round(2))

## 5. Sensitivity checks

Repeat the component estimates for dominance factors 2, 3 and 4 and minimum depths of 2,000, 3,000 and 4,000 m. A robust background direction should not depend on one arbitrary cutoff.

In [ ]:
rows = []
for factor in [2.0, 3.0, 4.0]:
    for depth in [2000.0, 3000.0, 4000.0]:
        cfg = PVAlignmentConfig(dominance_factor=factor, min_open_ocean_depth_m=depth, min_tilt_distance_km=MIN_TILT_km)
        use = add_pv_alignment_diagnostics(df, cfg)
        use = use[use.open_ocean_planetary & use.direction_valid].copy()
        use['x'] = use.TiltDis * np.sin(np.deg2rad(use.TiltDir))
        use['y'] = use.TiltDis * np.cos(np.deg2rad(use.TiltDir))
        means = use.groupby(['Cyc', 'Eddy'])[['x', 'y']].mean().groupby('Cyc').mean()
        for cyc in means.index:
            rows.append({'dominance_factor': factor, 'min_depth_m': depth, 'Cyc': cyc,
                         'mean_x_km': means.loc[cyc, 'x'], 'mean_y_km': means.loc[cyc, 'y'],
                         'eddies': use.loc[use.Cyc == cyc, 'Eddy'].nunique()})
display(pd.DataFrame(rows).round(2))